In [1]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
from typing import List

sys.path.append("..")
from src import create_sentence_nace_code_similarities, analysis_functions
import test_base
from sentence_splitter import split_text_into_sentences

## Test retrieving the similarities for chunks in a pdf to the NACE Code

**Function:** pdf-> (chunk x code -> [-1,1])

**Parameters:** 

- pdf_path
- way of chunking the text (e.g. sentences, sliding window, or paragraphs)
- way of preprocessing (most is fixed for all reports)
    - similarity threshold of relevant chunks
    - length of irrelevant chunks

**Store analytics for each datapoint:**

- mean score for each class given a threshold

In [2]:
# Parameters: 

threshold_min_chunk_len = 100
cos_threshold = 0.4
sentence_length = 6

In [3]:
dataset_path = "../data/datasets/german_annual_reports"
dataset_path = "../data/datasets/stoxx_600"
dataset_path = "../data/datasets/stoxx_600_extended"
dataset_path = "../data/datasets/reports_subset_from_full_data_1"

In [4]:
over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

In [5]:
nace_classes = pd.read_csv(over_view_df_path, index_col=0)
nace_classes.head()

,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,NACE,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
5789,ZW0009011041,Ariston Holdings Ltd.,1,1947.0,ZWE,V97772103,20090317.0,ZWE,@NA,1.19,...,ZW0009011041,603408,Ariston Holdings Ltd.,1.0,ARIS-ZW,1,SHARE,6034081,A,Ariston Holdings Ltd.1.pdf
35816,INE978A01027,Heritage Foods Limited,1,1992.0,IND,Y3179H146,20020117.0,IND,06FQLY-E,1.41,...,INE978A01027,BF2F40,Heritage Foods Limited,1.0,519552-IN,1,SHARE,BF2F405,A,Heritage Foods Limited1.pdf
80373,MYL7854OO002,Timberwell Bhd.,1,1996.0,MYS,Y88399103,19970516.0,MYS,05JH15-E,2.30,...,MYL7854OO002,690556,Timberwell Bhd.,1.0,7854-MY,1,SHARE,6905563,A,Timberwell Bhd.1.pdf
49813,MYQ0189OO009,Matang Bhd.,1,2015.0,MYS,Y58347108,20170117.0,MYS,@NA,1.19,...,MYQ0189OO009,BYYQB5,Matang Bhd.,1.0,0189-MY,1,SHARE,BYYQB53,A,Matang Bhd.2.pdf
73064,MYL4316OO005,Sin Heng Chan (Malaya) Bhd.,1,1962.0,MYS,Y80178109,19880324.0,MYS,05YMQ5-E,1.19,...,MYL4316OO005,681088,Sin Heng Chan (Malaya) Bhd.,1.0,4316-MY,1,SHARE,6810883,A,Sin Heng Chan (Malaya) Bhd.1.pdf


In [6]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class

{'Ariston Holdings Ltd.1.pdf': 1.19,
 'Heritage Foods Limited1.pdf': 1.41,
 'Timberwell Bhd.1.pdf': 2.3,
 'Matang Bhd.2.pdf': 1.19,
 'Sin Heng Chan (Malaya) Bhd.1.pdf': 1.19,
 'Tech-bank Food Co., Ltd.3.pdf': 1.46,
 'Namoi Cotton Ltd1.pdf': 1.63,
 'Atlantic Sapphire ASA1.pdf': 3.11,
 'Agra Limited2.pdf': 1.19,
 'Huisheng International Holdings Ltd.3.pdf': 1.46,
 'Australian Agricultural Company Limited1.pdf': 1.62,
 'Waterbase Limited2.pdf': 3.21,
 'CannAmerica Brands Corp.1.pdf': 1.3,
 'Qian Hu Corporation Limited1.pdf': 3.21,
 'Jawala Inc.1.pdf': 1.19,
 'Green Thumb Industries Inc.1.pdf': 1.19,
 'CLS Holdings USA Inc2.pdf': 1.19,
 'China Bozza Development Holdings Limited1.pdf': 2.4,
 'Salmon Evolution ASA1.pdf': 3.21,
 'North American Cannabis Holdings, Inc.1.pdf': 1.19,
 'Malwatte Valley Plantations Plc1.pdf': 1.61,
 'Greenheart Group Limited1.pdf': 2.2,
 'Bumitama Agri Ltd.1.pdf': 1.19,
 'Genus plc1.pdf': 1.62,
 'Kotagala Plantations Plc1.pdf': 2.3,
 'PT Andira Agro Tbk1.pdf': 1.1

In [7]:
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
report_to_nace_class

{'Ariston Holdings Ltd.1.txt': 1.19,
 'Heritage Foods Limited1.txt': 1.41,
 'Timberwell Bhd.1.txt': 2.3,
 'Matang Bhd.2.txt': 1.19,
 'Sin Heng Chan (Malaya) Bhd.1.txt': 1.19,
 'Tech-bank Food Co., Ltd.3.txt': 1.46,
 'Namoi Cotton Ltd1.txt': 1.63,
 'Atlantic Sapphire ASA1.txt': 3.11,
 'Agra Limited2.txt': 1.19,
 'Huisheng International Holdings Ltd.3.txt': 1.46,
 'Australian Agricultural Company Limited1.txt': 1.62,
 'Waterbase Limited2.txt': 3.21,
 'CannAmerica Brands Corp.1.txt': 1.3,
 'Qian Hu Corporation Limited1.txt': 3.21,
 'Jawala Inc.1.txt': 1.19,
 'Green Thumb Industries Inc.1.txt': 1.19,
 'CLS Holdings USA Inc2.txt': 1.19,
 'China Bozza Development Holdings Limited1.txt': 2.4,
 'Salmon Evolution ASA1.txt': 3.21,
 'North American Cannabis Holdings, Inc.1.txt': 1.19,
 'Malwatte Valley Plantations Plc1.txt': 1.61,
 'Greenheart Group Limited1.txt': 2.2,
 'Bumitama Agri Ltd.1.txt': 1.19,
 'Genus plc1.txt': 1.62,
 'Kotagala Plantations Plc1.txt': 2.3,
 'PT Andira Agro Tbk1.txt': 1.1

In [8]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
reports_path

['../data/datasets/reports_subset_from_full_data_1/TXTs/PVH Corp.3.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Mekdam Holding Group Company1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Ambea AB1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Harbour Equine Holdings Limited3.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Poste Italiane SpA1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Propel Funeral Partners Ltd.1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Nabors Industries Ltd.2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/MK Land Holdings Bhd.2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Brookfield Infrastructure Corp. (New York)2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Shangri-La Hotel Public Co. Ltd.2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Rentokil Initial plc1.txt',
 '../data/datasets/reports_subset_fro

In [9]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        lambda line: line == '<!-- image -->',
        
        #filter tables 
        lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        lambda line: "." not in line,
        
        # more than 50% is numbers
        lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [10]:
for i in range(4,5): 
    nace_level = i

    result_path = f"../results/dataset__{dataset_name}_sentence_len_{sentence_length}__min_chunk_len_{threshold_min_chunk_len}__cos_thresh_{cos_threshold}__nace_level_{nace_level}"

    res = test_base.test_similarities(reversed(reports_path), preprocess_report, threshold_min_chunk_len, cos_threshold, report_to_nace_class, result_path, level=i, overwrite=False)

0it [00:00, ?it/s]

{'L_REAL ESTATE ACTIVITIES': 0.03175, 'K_FINANCIAL AND INSURANCE ACTIVITIES': 0.023277777777777776, 'I_ACCOMMODATION AND FOOD SERVICE ACTIVITIES': 0.006875, 'M_PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIVITIES': 0.004631578947368421, 'N_ADMINISTRATIVE AND SUPPORT SERVICE ACTIVITIES': 0.0038484848484848485, 'F_CONSTRUCTION': 0.0025454545454545456, 'T_ACTIVITIES OF HOUSEHOLDS AS EMPLOYERS; UNDIFFERENTIATED GOODS- AND SERVICES-PRODUCING ACTIVITIES OF HOUSEHOLDS FOR OWN USE': 0.001, 'O_PUBLIC ADMINISTRATION AND DEFENCE; COMPULSORY SOCIAL SECURITY': 0.001, 'U_ACTIVITIES OF EXTRATERRITORIAL ORGANISATIONS AND BODIES': 0.001, 'G_WHOLESALE AND RETAIL TRADE; REPAIR OF MOTOR VEHICLES AND MOTORCYCLES': 0.001, 'S_OTHER SERVICE ACTIVITIES': 0.0005789473684210526, 'J_INFORMATION AND COMMUNICATION': 0.0005384615384615384, 'H_TRANSPORTATION AND STORAGE': 0.000391304347826087, 'P_EDUCATION': 0.00036363636363636367, 'R_ARTS, ENTERTAINMENT AND RECREATION': 0.0003333333333333333, 'A_AGRICULTURE, FORESTRY A

3it [01:00, 20.19s/it]

{'64.3_Trusts, funds and similar financial entities': 0.082, '64.2_Activities of holding companies': 0.06, '68.1_Buying and selling of own real estate': 0.055, '66.3_Fund management activities': 0.041, '64.9_Other financial service activities, except insurance and pension funding': 0.038, '68.2_Renting and operating of own or leased real estate': 0.032, '41.1_Development of building projects': 0.029, '70.1_Activities of head offices': 0.026, '68.3_Real estate activities on a fee or contract basis': 0.02, '69.2_Accounting, bookkeeping and auditing activities; tax consultancy': 0.02, '70.2_Management consultancy activities': 0.0185, '55.2_Holiday and other short-stay accommodation': 0.016, '55.1_Hotels and similar accommodation': 0.016, '65.3_Pension funding': 0.014, '66.1_Activities auxiliary to financial services, except insurance and pension funding': 0.013333333333333334, '65.2_Reinsurance': 0.013, '77.4_Leasing of intellectual property and similar products, except copyrighted works'

10it [02:08, 12.14s/it]

...

{'64.2_Activities of holding companies': 0.073, '64.3_Trusts, funds and similar financial entities': 0.059, '70.1_Activities of head offices': 0.034, '65.3_Pension funding': 0.034, '66.3_Fund management activities': 0.033, '41.1_Development of building projects': 0.028, '70.2_Management consultancy activities': 0.027499999999999997, '68.1_Buying and selling of own real estate': 0.027, '64.9_Other financial service activities, except insurance and pension funding': 0.02466666666666667, '77.4_Leasing of intellectual property and similar products, except copyrighted works': 0.022, '68.2_Renting and operating of own or leased real estate': 0.019, '69.2_Accounting, bookkeeping and auditing activities; tax consultancy': 0.017, '94.2_Activities of trade unions': 0.016, '01.5_Mixed farming': 0.015, '81.1_Combined facilities support activities': 0.014, '65.2_Reinsurance': 0.014, '66.1_Activities auxiliary to financial services, except insurance and pension funding': 0.013, '78.3_Other human res

164it [2:35:17, 31.11s/it]

{'64.2_Activities of holding companies': 0.068, '64.3_Trusts, funds and similar financial entities': 0.065, '65.3_Pension funding': 0.053, '66.3_Fund management activities': 0.043, '64.9_Other financial service activities, except insurance and pension funding': 0.04, '70.2_Management consultancy activities': 0.038000000000000006, '69.2_Accounting, bookkeeping and auditing activities; tax consultancy': 0.036, '01.5_Mixed farming': 0.036, '77.4_Leasing of intellectual property and similar products, except copyrighted works': 0.025, '70.1_Activities of head offices': 0.022, '64.1_Monetary intermediation': 0.021, '84.3_Compulsory social security activities': 0.019, '82.1_Office administrative and support activities': 0.017, '66.1_Activities auxiliary to financial services, except insurance and pension funding': 0.016666666666666666, '46.9_Non-specialised wholesale trade': 0.016, '41.1_Development of building projects': 0.013, '66.2_Activities auxiliary to insurance and pension funding': 0.

169it [2:41:21, 57.29s/it]


OSError: [Errno 28] No space left on device: '../results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_4/MONTEA NV1.txt/relevant_sentences_MONTEA NV1.txt'